In [6]:
# =============================================================================
# NOTEBOOK:  01_transform_bronze_to_silver
# ATTACH TO: lakehouse_bronze
# RUN AFTER: Steps 1-4 complete (all Bronze data loaded)
#            Step 5a complete (shortcuts from lakehouse_bronze created in lakehouse_silver)
# =============================================================================
# WHY ATTACH TO BRONZE:
#   - We need to read Files/raw/online_orders.json (relative path only works
#     for the attached lakehouse)
#   - We read Bronze tables via: lakehouse_bronze.dbo.table_name
#   - We write Silver tables via: lakehouse_silver.dbo.table_name
#     (Spark can write to any lakehouse using full 3-part name)
#
# SCHEMA RULES:
#   Read from Bronze:  lakehouse_bronze.dbo.table_name  (3-part)
#   Write to Silver:   lakehouse_silver.dbo.table_name  (3-part)
#   Read JSON file:    Files/raw/online_orders.json     (relative, attached LH)


StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 8, Finished, Available, Finished, False)

In [7]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, DoubleType, StringType, DateType
)
from pyspark.sql import Row
from datetime import date, timedelta

print("Imports loaded.")

StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 9, Finished, Available, Finished, False)

Imports loaded.


In [8]:
df_customers = spark.sql("SELECT * FROM aussie_brew_lakehouse.dbo.customers")
df_employees = spark.sql("SELECT * FROM aussie_brew_lakehouse.dbo.employees")
df_sales     = spark.sql("SELECT * FROM aussie_brew_lakehouse.dbo.sales")
df_products  = spark.sql("SELECT * FROM aussie_brew_lakehouse.dbo.products")
df_stores    = spark.sql("SELECT * FROM aussie_brew_lakehouse.dbo.stores")

# JSON from Files section (relative path works because we are attached to lakehouse_bronze)
df_online_raw = spark.read.option("multiline", "true").json("Files/raw/online_orders.json")

print("Bronze tables loaded:")
print(f"  customers:     {df_customers.count()} rows")
print(f"  employees:     {df_employees.count()} rows")
print(f"  sales:         {df_sales.count()} rows")
print(f"  products:      {df_products.count()} rows")
print(f"  stores:        {df_stores.count()} rows")
print(f"  online (JSON): {df_online_raw.count()} rows")

StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 10, Finished, Available, Finished, False)

Bronze tables loaded:
  customers:     50 rows
  employees:     24 rows
  sales:         420 rows
  products:      15 rows
  stores:        6 rows
  online (JSON): 80 rows


In [9]:
# =============================================================================
# CELL 3 — Save JSON as Bronze table
# =============================================================================

# JSON has nested "items" array. Save raw first (Bronze = no changes).
# Notebook is attached to lakehouse_bronze, so short name works too.

df_online_raw.write.mode("overwrite").format("delta").saveAsTable(
    "aussie_brew_lakehouse.dbo.online_orders"
)

print(f"online_orders table created in lakehouse_bronze: {df_online_raw.count()} rows")
print("Bronze layer complete — 6 tables in lakehouse_bronze.")

StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 11, Finished, Available, Finished, False)

online_orders table created in lakehouse_bronze: 80 rows
Bronze layer complete — 6 tables in lakehouse_bronze.


In [10]:
# =============================================================================
# CELL 4 — Python vs SQL Demo: Store Performance Analysis
# =============================================================================
# For each store, for each month:
#   Revenue, transaction count, rank, MoM change, cumulative, alert flag
#
# APPROACH 1: SQL (~70 lines)

print("=" * 60)
print("APPROACH 1: SQL (~70 lines)")
print("=" * 60)

df_sql_result = spark.sql("""
    WITH monthly_store_revenue AS (
        SELECT
            s.store_id,
            s.store_name,
            s.city,
            s.state,
            CONCAT(
                CAST(YEAR(CAST(f.transaction_date AS DATE)) AS STRING), '-',
                LPAD(CAST(MONTH(CAST(f.transaction_date AS DATE)) AS STRING), 2, '0')
            ) AS year_month,
            MONTH(CAST(f.transaction_date AS DATE)) AS month_number,
            COUNT(DISTINCT f.transaction_id) AS transaction_count,
            ROUND(SUM(f.total_amount), 2) AS revenue
        FROM aussie_brew_lakehouse.dbo.sales f
        JOIN aussie_brew_lakehouse.dbo.stores s
            ON f.store_id = s.store_id
        GROUP BY
            s.store_id, s.store_name, s.city, s.state,
            CONCAT(
                CAST(YEAR(CAST(f.transaction_date AS DATE)) AS STRING), '-',
                LPAD(CAST(MONTH(CAST(f.transaction_date AS DATE)) AS STRING), 2, '0')
            ),
            MONTH(CAST(f.transaction_date AS DATE))
    ),

    ranked AS (
        SELECT
            store_id, store_name, city, state,
            year_month, month_number, transaction_count, revenue,
            RANK() OVER (
                PARTITION BY year_month ORDER BY revenue DESC
            ) AS revenue_rank,
            LAG(revenue) OVER (
                PARTITION BY store_id ORDER BY year_month
            ) AS prev_month_revenue,
            SUM(revenue) OVER (
                PARTITION BY store_id ORDER BY year_month
                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
            ) AS cumulative_revenue
        FROM monthly_store_revenue
    )

    SELECT
        store_id, store_name, city, state, year_month,
        transaction_count, revenue, revenue_rank, prev_month_revenue,
        ROUND(revenue - COALESCE(prev_month_revenue, 0), 2) AS mom_change,
        ROUND(
            CASE
                WHEN prev_month_revenue IS NULL OR prev_month_revenue = 0 THEN NULL
                ELSE (revenue - prev_month_revenue) / prev_month_revenue * 100
            END, 1
        ) AS mom_change_pct,
        cumulative_revenue,
        CASE
            WHEN prev_month_revenue IS NOT NULL
             AND revenue < prev_month_revenue * 0.9
            THEN 'ALERT'
            ELSE 'OK'
        END AS performance_flag
    FROM ranked
    ORDER BY year_month, revenue_rank
""")

df_sql_result.show(12, truncate=False)

StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 12, Finished, Available, Finished, False)

APPROACH 1: SQL (~70 lines)
+--------+-------------------------------+---------+-----+----------+-----------------+-------+------------+------------------+----------+--------------+------------------+----------------+
|store_id|store_name                     |city     |state|year_month|transaction_count|revenue|revenue_rank|prev_month_revenue|mom_change|mom_change_pct|cumulative_revenue|performance_flag|
+--------+-------------------------------+---------+-----+----------+-----------------+-------+------------+------------------+----------+--------------+------------------+----------------+
|STR005  |Aussie Brew - Adelaide Rundle  |Adelaide |SA   |2025-01   |39               |584.52 |1           |NULL              |584.52    |NULL          |584.52            |OK              |
|STR006  |Aussie Brew - Canberra Centre  |Canberra |ACT  |2025-01   |35               |534.74 |2           |NULL              |534.74    |NULL          |534.74            |OK              |
|STR004  |Aussie Brew 

In [11]:
# APPROACH 2: Python (~10 lines of logic)

print("=" * 60)
print("APPROACH 2: Python (~10 lines of logic)")
print("=" * 60)

df_monthly = (
    df_sales.join(df_stores, "store_id")
    .withColumn("year_month", F.date_format(F.to_date("transaction_date"), "yyyy-MM"))
    .withColumn("month_number", F.month(F.to_date("transaction_date")))
    .groupBy("store_id", "store_name", "city", "state", "year_month", "month_number")
    .agg(
        F.countDistinct("transaction_id").alias("transaction_count"),
        F.round(F.sum("total_amount"), 2).alias("revenue")
    )
)

w_rank  = Window.partitionBy("year_month").orderBy(F.desc("revenue"))
w_store = Window.partitionBy("store_id").orderBy("year_month")
w_cumul = w_store.rowsBetween(Window.unboundedPreceding, Window.currentRow)

df_python_result = (
    df_monthly
    .withColumn("revenue_rank", F.rank().over(w_rank))
    .withColumn("prev_month_revenue", F.lag("revenue").over(w_store))
    .withColumn("cumulative_revenue", F.sum("revenue").over(w_cumul))
    .withColumn("mom_change", F.round(F.col("revenue") - F.coalesce(F.col("prev_month_revenue"), F.lit(0)), 2))
    .withColumn("mom_change_pct", F.round((F.col("revenue") - F.col("prev_month_revenue")) / F.col("prev_month_revenue") * 100, 1))
    .withColumn("performance_flag", F.when(F.col("revenue") < F.col("prev_month_revenue") * 0.9, "ALERT").otherwise("OK"))
    .orderBy("year_month", "revenue_rank")
)

df_python_result.show(12, truncate=False)

print("""
WHY PYTHON WINS:
  1. Window definitions are REUSABLE OBJECTS (define once, use many)
  2. Method chaining reads top-to-bottom like a recipe
  3. No CTEs — each .withColumn() adds one calculation
  4. Same Spark engine — identical performance
  5. Easy to add/remove columns — just one line
""")

StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 13, Finished, Available, Finished, False)

APPROACH 2: Python (~10 lines of logic)
+--------+-------------------------------+---------+-----+----------+------------+-----------------+-------+------------+------------------+------------------+----------+--------------+----------------+
|store_id|store_name                     |city     |state|year_month|month_number|transaction_count|revenue|revenue_rank|prev_month_revenue|cumulative_revenue|mom_change|mom_change_pct|performance_flag|
+--------+-------------------------------+---------+-----+----------+------------+-----------------+-------+------------+------------------+------------------+----------+--------------+----------------+
|STR005  |Aussie Brew - Adelaide Rundle  |Adelaide |SA   |2025-01   |1           |39               |584.52 |1           |NULL              |584.52            |584.52    |NULL          |OK              |
|STR006  |Aussie Brew - Canberra Centre  |Canberra |ACT  |2025-01   |1           |35               |534.74 |2           |NULL              |534.74  

In [12]:
# =============================================================================
# CELL 5 — dim_date
# =============================================================================

start_date = date(2024, 1, 1)
end_date   = date(2026, 12, 31)
date_list  = []
d = start_date
while d <= end_date:
    date_list.append(Row(full_date=d))
    d += timedelta(days=1)

df_dim_date = spark.createDataFrame(date_list)
df_dim_date = df_dim_date.select(
    F.date_format("full_date", "yyyyMMdd").cast(IntegerType()).alias("date_key"),
    F.col("full_date"),
    F.dayofmonth("full_date").alias("day_of_month"),
    F.date_format("full_date", "EEEE").alias("day_name"),
    F.dayofweek("full_date").alias("day_of_week"),
    F.date_format("full_date", "MMMM").alias("month_name"),
    F.month("full_date").alias("month_number"),
    F.quarter("full_date").alias("quarter"),
    F.year("full_date").alias("year"),
    F.when(F.dayofweek("full_date").isin(1, 7), "Yes").otherwise("No").alias("is_weekend"),
    F.date_format("full_date", "yyyy-MM").alias("year_month")
)

print(f"dim_date: {df_dim_date.count()} rows")
df_dim_date.show(5)

StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 14, Finished, Available, Finished, False)

dim_date: 1096 rows
+--------+----------+------------+---------+-----------+----------+------------+-------+----+----------+----------+
|date_key| full_date|day_of_month| day_name|day_of_week|month_name|month_number|quarter|year|is_weekend|year_month|
+--------+----------+------------+---------+-----------+----------+------------+-------+----+----------+----------+
|20240101|2024-01-01|           1|   Monday|          2|   January|           1|      1|2024|        No|   2024-01|
|20240102|2024-01-02|           2|  Tuesday|          3|   January|           1|      1|2024|        No|   2024-01|
|20240103|2024-01-03|           3|Wednesday|          4|   January|           1|      1|2024|        No|   2024-01|
|20240104|2024-01-04|           4| Thursday|          5|   January|           1|      1|2024|        No|   2024-01|
|20240105|2024-01-05|           5|   Friday|          6|   January|           1|      1|2024|        No|   2024-01|
+--------+----------+------------+---------+--------

In [13]:
# =============================================================================
# CELL 6 — dim_store
# =============================================================================

w = Window.orderBy("store_id")
df_dim_store = df_stores.select(
    F.row_number().over(w).alias("store_key"),
    F.col("store_id"),
    F.trim(F.col("store_name")).alias("store_name"),
    F.initcap(F.trim(F.col("city"))).alias("city"),
    F.upper(F.trim(F.col("state"))).alias("state"),
    F.col("postcode"),
    F.initcap(F.trim(F.col("manager"))).alias("manager"),
    F.to_date(F.col("open_date")).alias("open_date"),
    F.col("store_type")
)

print(f"dim_store: {df_dim_store.count()} rows")
df_dim_store.show()

StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 15, Finished, Available, Finished, False)

dim_store: 6 rows
+---------+--------+--------------------+---------+-----+--------+--------------+----------+----------+
|store_key|store_id|          store_name|     city|state|postcode|       manager| open_date|store_type|
+---------+--------+--------------------+---------+-----+--------+--------------+----------+----------+
|        1|  STR001|Aussie Brew - Syd...|   Sydney|  NSW|    2000|Sarah Mitchell|2020-03-15|  Flagship|
|        2|  STR002|Aussie Brew - Mel...|Melbourne|  VIC|    3000|    James Wong|2020-06-01|  Flagship|
|        3|  STR003|Aussie Brew - Bri...| Brisbane|  QLD|    4000|   Emily Davis|2021-01-10|  Standard|
|        4|  STR004|Aussie Brew - Per...|    Perth|   WA|    6000|  Michael Chen|2021-08-20|  Standard|
|        5|  STR005|Aussie Brew - Ade...| Adelaide|   SA|    5000|     Lisa Park|2022-02-14|     Kiosk|
|        6|  STR006|Aussie Brew - Can...| Canberra|  ACT|    2601|     Tom Brown|2022-09-01|     Kiosk|
+---------+--------+--------------------+-----

In [14]:
# =============================================================================
# CELL 7 — dim_product
# =============================================================================

w = Window.orderBy("product_id")
df_dim_product = df_products.select(
    F.row_number().over(w).alias("product_key"),
    F.col("product_id"),
    F.initcap(F.trim(F.col("product_name"))).alias("product_name"),
    F.initcap(F.trim(F.col("category"))).alias("category"),
    F.initcap(F.trim(F.col("subcategory"))).alias("subcategory"),
    F.col("unit_price").cast(DoubleType()),
    F.col("cost_price").cast(DoubleType()),
    F.round(
        (F.col("unit_price") - F.col("cost_price")) / F.col("unit_price") * 100, 1
    ).alias("profit_margin_pct"),
    F.col("is_active")
)

print(f"dim_product: {df_dim_product.count()} rows")
df_dim_product.show(5, truncate=False)

StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 16, Finished, Available, Finished, False)

dim_product: 15 rows
+-----------+----------+------------+--------+-----------+----------+----------+-----------------+---------+
|product_key|product_id|product_name|category|subcategory|unit_price|cost_price|profit_margin_pct|is_active|
+-----------+----------+------------+--------+-----------+----------+----------+-----------------+---------+
|1          |PRD001    |Flat White  |Coffee  |Espresso   |5.5       |1.8       |67.3             |Y        |
|2          |PRD002    |Cappuccino  |Coffee  |Espresso   |5.5       |1.8       |67.3             |Y        |
|3          |PRD003    |Long Black  |Coffee  |Espresso   |5.0       |1.5       |70.0             |Y        |
|4          |PRD004    |Latte       |Coffee  |Espresso   |5.5       |1.8       |67.3             |Y        |
|5          |PRD005    |Mocha       |Coffee  |Specialty  |6.5       |2.2       |66.2             |Y        |
+-----------+----------+------------+--------+-----------+----------+----------+-----------------+---------

In [15]:
# =============================================================================
# CELL 8 — dim_customer
# =============================================================================

w = Window.orderBy("customer_id")
df_dim_customer = df_customers.select(
    F.row_number().over(w).alias("customer_key"),
    F.col("customer_id"),
    F.concat(
        F.initcap(F.trim(F.col("first_name"))),
        F.lit(" "),
        F.initcap(F.trim(F.col("last_name")))
    ).alias("full_name"),
    F.lower(F.trim(F.col("email"))).alias("email"),
    F.col("phone"),
    F.initcap(F.trim(F.col("suburb"))).alias("suburb"),
    F.upper(F.trim(F.col("state"))).alias("state"),
    F.initcap(F.trim(F.col("loyalty_tier"))).alias("loyalty_tier"),
    F.to_date(F.col("signup_date")).alias("signup_date"),
    F.col("total_lifetime_spend").cast(DoubleType())
)

# FIX: Explicit schema required because signup_date=None
# Spark cannot infer type from None — must define StructType

unknown_schema = StructType([
    StructField("customer_key", IntegerType(), False),
    StructField("customer_id", StringType(), False),
    StructField("full_name", StringType(), False),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("suburb", StringType(), True),
    StructField("state", StringType(), True),
    StructField("loyalty_tier", StringType(), True),
    StructField("signup_date", DateType(), True),
    StructField("total_lifetime_spend", DoubleType(), True),
])

unknown = spark.createDataFrame(
    [(0, "UNKNOWN", "Walk-in Customer", "", "", "Unknown", "Unknown", "None", None, 0.0)],
    schema=unknown_schema
)

df_dim_customer = df_dim_customer.unionByName(unknown)

print(f"dim_customer: {df_dim_customer.count()} rows (includes Walk-in row with key=0)")


StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 17, Finished, Available, Finished, False)

dim_customer: 51 rows (includes Walk-in row with key=0)


In [16]:
# =============================================================================
# CELL 9 — dim_employee
# =============================================================================

w = Window.orderBy("employee_id")
df_dim_employee = df_employees.select(
    F.row_number().over(w).alias("employee_key"),
    F.col("employee_id"),
    F.initcap(F.trim(F.col("employee_name"))).alias("employee_name"),
    F.initcap(F.trim(F.col("role"))).alias("role"),
    F.col("store_id"),
    F.col("hourly_rate").cast(DoubleType()),
    F.col("weekly_hours").cast(IntegerType()),
    F.to_date(F.col("start_date")).alias("start_date"),
    F.when(F.col("is_active") == "Y", "Active").otherwise("Inactive").alias("status"),
    F.round(F.col("hourly_rate") * F.col("weekly_hours") * 52, 2).alias("estimated_annual_salary")
)

print(f"dim_employee: {df_dim_employee.count()} rows")



StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 18, Finished, Available, Finished, False)

dim_employee: 24 rows


In [17]:
# =============================================================================
# CELL 10 — fact_sales
# =============================================================================

df_fact_sales = (
    df_sales.alias("s")
    .join(
        df_dim_store.select("store_id", "store_key").alias("st"),
        F.col("s.store_id") == F.col("st.store_id"), "left"
    )
    .join(
        df_dim_product.select("product_id", "product_key").alias("p"),
        F.col("s.product_id") == F.col("p.product_id"), "left"
    )
    .join(
        df_dim_customer.select("customer_id", "customer_key").alias("c"),
        F.col("s.customer_id") == F.col("c.customer_id"), "left"
    )
    .select(
        F.col("s.transaction_id"),
        F.date_format(F.to_date(F.col("s.transaction_date")), "yyyyMMdd").cast(IntegerType()).alias("date_key"),
        F.col("st.store_key").cast(IntegerType()),
        F.col("p.product_key").cast(IntegerType()),
        F.when(F.col("c.customer_key").isNull(), F.lit(0))
            .otherwise(F.col("c.customer_key")).cast(IntegerType()).alias("customer_key"),
        F.col("s.transaction_time"),
        F.col("s.quantity").cast(IntegerType()),
        F.col("s.unit_price").cast(DoubleType()),
        F.col("s.discount_pct").cast(DoubleType()),
        F.col("s.total_amount").cast(DoubleType()),
        F.col("s.payment_method")
    )
)

print(f"fact_sales: {df_fact_sales.count()} rows")

StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 19, Finished, Available, Finished, False)

fact_sales: 420 rows


In [18]:
# =============================================================================
# CELL 11 — fact_online_orders (flatten nested JSON)
# =============================================================================

df_online_exploded = (
    df_online_raw
    .select(
        F.col("order_id"),
        F.to_date(F.col("order_date")).alias("order_date"),
        F.col("customer_id"),
        F.col("order_channel"),
        F.col("delivery_suburb"),
        F.col("delivery_state"),
        F.col("order_total").cast(DoubleType()),
        F.col("delivery_fee").cast(DoubleType()),
        F.col("payment_method"),
        F.col("order_status"),
        F.explode(F.col("items")).alias("item")
    )
    .select(
        F.col("order_id"),
        F.date_format(F.col("order_date"), "yyyyMMdd").cast(IntegerType()).alias("date_key"),
        F.col("customer_id"),
        F.col("order_channel"),
        F.col("item.product_id").alias("product_id"),
        F.col("item.product_name").alias("product_name"),
        F.col("item.quantity").cast(IntegerType()).alias("quantity"),
        F.col("item.unit_price").cast(DoubleType()).alias("unit_price"),
        F.col("item.subtotal").cast(DoubleType()).alias("line_total"),
        F.col("order_total"),
        F.col("delivery_fee"),
        F.col("delivery_suburb"),
        F.col("delivery_state"),
        F.col("payment_method"),
        F.col("order_status")
    )
)

df_fact_online = (
    df_online_exploded.alias("o")
    .join(
        df_dim_customer.select("customer_id", "customer_key").alias("c"),
        F.col("o.customer_id") == F.col("c.customer_id"), "left"
    )
    .join(
        df_dim_product.select("product_id", "product_key").alias("p"),
        F.col("o.product_id") == F.col("p.product_id"), "left"
    )
    .select(
        F.col("o.order_id"),
        F.col("o.date_key"),
        F.when(F.col("c.customer_key").isNull(), F.lit(0))
            .otherwise(F.col("c.customer_key")).cast(IntegerType()).alias("customer_key"),
        F.col("p.product_key").cast(IntegerType()),
        F.col("o.order_channel"),
        F.col("o.product_name"),
        F.col("o.quantity"),
        F.col("o.unit_price"),
        F.col("o.line_total"),
        F.col("o.order_total"),
        F.col("o.delivery_fee"),
        F.col("o.delivery_suburb"),
        F.col("o.delivery_state"),
        F.col("o.payment_method"),
        F.col("o.order_status")
    )
)

print(f"fact_online_orders: {df_fact_online.count()} rows")

StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 20, Finished, Available, Finished, False)

fact_online_orders: 197 rows


In [19]:
# =============================================================================
# CELL 12 — Write all Silver tables to aussie_brew_lakehouse
# =============================================================================

tables = {
    "dim_date":           df_dim_date,
    "dim_store":          df_dim_store,
    "dim_product":        df_dim_product,
    "dim_customer":       df_dim_customer,
    "dim_employee":       df_dim_employee,
    "fact_sales":         df_fact_sales,
    "fact_online_orders": df_fact_online,
}

for name, df in tables.items():
    df.write.mode("overwrite").format("delta").saveAsTable(
        f"aussie_brew_lakehouse.dbo.{name}"
    )
    print(f"  Written: aussie_brew_lakehouse.dbo.{name}")

print()
print("All 7 Silver tables written to aussie_brew_lakehouse.")

StatementMeta(, 7ef7226b-6702-444f-b3a2-e0d83bbc7242, 21, Finished, Available, Finished, False)

  Written: aussie_brew_lakehouse.dbo.dim_date
  Written: aussie_brew_lakehouse.dbo.dim_store
  Written: aussie_brew_lakehouse.dbo.dim_product
  Written: aussie_brew_lakehouse.dbo.dim_customer
  Written: aussie_brew_lakehouse.dbo.dim_employee
  Written: aussie_brew_lakehouse.dbo.fact_sales
  Written: aussie_brew_lakehouse.dbo.fact_online_orders

All 7 Silver tables written to aussie_brew_lakehouse.
